# Phase 1 — vocal input validation and acoustic analysis
**Owner:** Liyanage L. N. P. (IT23141506), J26-DS-339.

This notebook imports the maintained package and can run from a fresh kernel with **Run All**. It demonstrates original preservation, supported technical input, controlled stops, measured pitch and aligned features. Technical acceptance does not establish musical usability or intended notes. No BPDR repair model is running.

Use `Run-Notebooks.ps1` to select the component-local uv kernel. Raw data stays read-only; new outputs go to ignored `runs/`.

In [ ]:
import os, json, sys
from pathlib import Path
from collections import Counter
import numpy as np
import soundfile as sf
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
import bpdr_input
COMPONENT = Path(bpdr_input.__file__).resolve().parents[2]
DATASET = Path(os.environ.get('BPDR_DATASET_ROOT', str(COMPONENT.parents[2]/'Input Validation'/'vocadito')))
assert DATASET.is_dir(), 'Set BPDR_DATASET_ROOT to your vocadito folder before running.'
print('Python:', sys.version.split()[0], '\nComponent:', COMPONENT, '\nDataset:', DATASET)
plt.rcParams.update({'figure.figsize': (12, 4)})

## 1. Dataset inventory — recordings are the denominator

In [ ]:
from bpdr_input.inventory import audit_vocadito
inventory = audit_vocadito(DATASET)
display(inventory['summary'])

## 2. Process a full supported recording — no implicit cropping

In [ ]:
from bpdr_input.pipeline import process, validate_package, checksum
from bpdr_input.contracts import Status
supported = sorted(p for p in (DATASET/'Audio').glob('*.wav') if not p.name.startswith('._') and .25 <= sf.info(p).duration <= 20)
source = supported[0]
result, RUN = process(source, COMPONENT/'runs'/'notebook_phase1', COMPONENT/'checkpoints')
assert result.status == Status.ACCEPT_UNCHANGED, result.to_dict()
accepted = validate_package(result)
diagnostics = json.loads((RUN/'diagnostics.json').read_text())
assert checksum(source) == checksum(RUN/'original.wav')
assert abs(accepted.duration_seconds-sf.info(source).duration) <= 1/accepted.sample_rate_hz
display(result.to_dict())
display(diagnostics['operations'])
display(diagnostics['analysis'])

## 3. Listen to original and working copy

In [ ]:
display(Markdown('**Original (unchanged file bytes)**'))
display(Audio(filename=str(RUN/'original.wav')))
display(Markdown('**Accepted mono working copy — no pitch correction**'))
display(Audio(filename=accepted.path))

## 4. Inspect performed pitch and uncertainty

In [ ]:
with np.load(RUN/'features.npz') as archive:
    features = {k: archive[k] for k in archive.files}
t = features['times_seconds']
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ax[0].plot(t, features['f0_hz']); ax[0].set_ylabel('Measured F0 (Hz)')
ax[1].plot(t, features['confidence']); ax[1].plot(t, features['voiced_mask'], alpha=.5); ax[1].set_ylabel('Confidence / voiced')
ax[2].plot(t, features['provisional_boundary_score']); ax[2].set_ylabel('Provisional boundary'); ax[2].set_xlabel('Original time (seconds)')
fig.tight_layout(); plt.show()
assert np.isnan(features['f0_hz'][~features['voiced_mask']]).all()
assert np.isfinite(features['boundary_features_71']).all()
print('Feature shapes:', features['frame_features_70'].shape, features['boundary_features_71'].shape)

Spikes and missing F0 may be analyzer errors. Performed F0 is not intended melody. Unvoiced frames are NaN in plots and masked zeros in model features. Provisional boundaries are heuristics. Frame-only inputs have 70 features; timing-boundary metadata adds the 71st.

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
image = ax.imshow(features['log_mel_db'].T, origin='lower', aspect='auto', extent=[t[0], t[-1], 0, 64])
ax.set_xlabel('Original time (seconds)'); ax.set_ylabel('Mel band'); fig.colorbar(image, ax=ax, label='Power (dB)'); plt.show()

## 5. Controlled stops cannot feed downstream processing

In [ ]:
bad = RUN/'silence_fixture.wav'
sf.write(bad, np.zeros(22050), 22050)
silence, silence_run = process(bad, COMPONENT/'runs'/'notebook_phase1', COMPONENT/'checkpoints')
assert silence.status == Status.RERECORD_REQUIRED and silence.accepted_audio is None
corrupt = RUN/'corrupt_fixture.wav'; corrupt.write_bytes(b'not a WAV')
failure, failure_run = process(corrupt, COMPONENT/'runs'/'notebook_phase1', COMPONENT/'checkpoints')
assert failure.status == Status.TECHNICAL_FAILURE and failure.accepted_audio is None
for stopped in (silence, failure):
    try: validate_package(stopped)
    except ValueError: pass
    else: raise AssertionError('Rejected audio reached the consumer')
display([silence.to_dict(), failure.to_dict()])

## 6. Saved full-recording results and limitations

In [ ]:
summary_path = COMPONENT/'runs'/'phase1_dataset'/'summary.json'
if summary_path.exists():
    from bpdr_input.pipeline import Policy
    saved = json.loads(summary_path.read_text())
    current = {Path(row['audio_path']).name: row for row in inventory['recordings']}
    saved_rows = saved.get('recordings', [])
    matching = (saved.get('recording_count') == len(current)
                and {row.get('source') for row in saved_rows} == set(current)
                and dict(Counter(row.get('status') for row in saved_rows)) == saved.get('statuses'))
    if matching:
        try:
            for row in saved_rows:
                diagnostics = json.loads((Path(row['run_directory'])/'diagnostics.json').read_text())
                source = diagnostics['source']
                matching &= (Path(source['path']).resolve() == Path(current[row['source']]['audio_path']).resolve()
                             and source['sha256'] == current[row['source']]['sha256']
                             and diagnostics['policy']['version'] == Policy().version
                             and diagnostics['result']['status'] == row['status'])
        except (OSError, ValueError, KeyError, TypeError):
            matching = False
    if matching:
        display({k:v for k,v in saved.items() if k != 'recordings'})
        print('Saved replay matches the current recording bytes and Phase 1 policy; it is not a new accuracy study.')
        print('This older replay does not record a source-code hash, so code-revision equivalence is unverified.')
    else:
        display(Markdown('**Historical replay only:** its recording hashes, paths, results or policy do not match this dataset. Regenerate the full replay before citing its counts.'))
else:
    print('No full-replay artifact here. Run the README dataset verification command; no counts are assumed.')


**What this demonstrates:** usable technical processing, failure propagation, provenance and acoustic analysis (foundations of FR1/FR2/FR6/FR7).

**What remains:** human usability review, calibrated acceptance, local repair, BPDR response losses, model comparisons and team integration. The Phase 2 notebook creates the controlled experiment; neither notebook establishes a training advantage.